# Multi-Session LeJEPA Experiment Runner

Train and evaluate the LeJEPA teacher model on multi-session neural spike data.

**Order of operations:**
1. Set `CONFIG_PATH` in the Config cell.
2. Run all cells top to bottom.
3. Results (CSV + plots) are saved under `results_out_path` from your config.
4. Training curves, test metrics, and latent-space UMAP also render inline below.

**Prerequisite:** Run the dataset pipeline first (`experiments/data/create_dataset.py`) so a Parquet exists at your config's `data_path`.

Clone Repo

In [ ]:
!git clone https://github.com/jacobposchl/jepsyn
!pip install torch_brain==0.1.0
!pip install -q snntorch temporaldata umap-learn pyarrow pyyaml

In [ ]:
import os
import sys
from pathlib import Path
from google.colab import drive, files


# The repo is cloned to /content/jepsyn in Colab.
REPO_ROOT = Path("/content/jepsyn")
os.chdir(REPO_ROOT)
print(f"CWD: {Path.cwd()}")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "experiments" / "multi_session"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

from multi_session import load_and_prepare_data, train_lejepa, train_mae, load_checkpoint, distill_snn
from jepsyn.utils import evaluate_model, identify_units, save_results, verify_config
from jepsyn.plots import latent_space

from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge

import torch
from torch import device


print("Imports OK")

## Configuration

Point `CONFIG_PATH` at your experiment YAML. See `configs/lejepa_lif_visual_cortex.yaml` for a full template with all encoder, predictor, and training fields.

At minimum, `results_out_path` must be set in the config for checkpoints and plots to be saved to disk.

In [ ]:
drive.mount('/content/drive')

# Create a shortcut to the shared folder in your drive if its not in your drive already
# Define the path to your shortcut
# Change 'SNN-LeJEPA' if you named your shortcut differently
SHARED_ZIP_PATH = "/content/drive/MyDrive/SNN-LeJEPA/Data/Model Results/LeJEPA_export.zip"
LOCAL_EXTRACT_DIR = "/content/lejepa_model"

# Unzip directly to local runtime storage
if os.path.exists(SHARED_ZIP_PATH):
    !mkdir -p {LOCAL_EXTRACT_DIR}
    !unzip -o "{SHARED_ZIP_PATH}" -d {LOCAL_EXTRACT_DIR}
    print("✓ Weights extracted to local runtime.")
    
    # Verify the file exists
    ckpt_file = Path(LOCAL_EXTRACT_DIR) / "results/lejepa_checkpoint.pt"
    if ckpt_file.exists():
        print(f"✓ Found checkpoint at: {ckpt_file}")
    else:
        print("! Warning: lejepa_checkpoint.pt not found. Check zip contents:")
        !ls {LOCAL_EXTRACT_DIR}
else:
    print("Error: Shortcut not found. Check Step 1 or the file path.")
    

DRIVE_PARQUET_PATH = "/content/drive/MyDrive/SNN-LeJEPA/Data/Processed Datasets/Visual Cortex/vis_all_animals_dataset.parquet"

# Local destination where the script expects it
LOCAL_DATA_DIR = "/content/jepsyn/datasets"
LOCAL_PARQUET_PATH = f"{LOCAL_DATA_DIR}/vis_all_animals_dataset.parquet"

# Create directory and copy
if os.path.exists(DRIVE_PARQUET_PATH):
    !mkdir -p {LOCAL_DATA_DIR}
    !cp "{DRIVE_PARQUET_PATH}" "{LOCAL_PARQUET_PATH}"
    print(f"Dataset copied to local runtime: {LOCAL_PARQUET_PATH}")
else:
    print("Error: Parquet file not found on Drive. Double-check your shortcut and path.")
    # Quick debug: list the contents of the Drive folder
    !ls "/content/drive/MyDrive/SNN-LeJEPA/Data/Processed Datasets/Visual Cortex/"

In [ ]:
# Point CONFIG_PATH at your experiment YAML to select the model.
# e.g. lejepa_visual_cortex.yaml (SIGReg), vicreg_visual_cortex.yaml (VICReg)
CONFIG_PATH = Path("experiments/multi_session/configs/lejepa_visual_cortex.yaml")

config = verify_config(CONFIG_PATH)

tcfg = config["training_config"]
mcfg = config["model_config"]
reg_type = tcfg.get("reg_type", "sigreg").lower()

print(f"Config loaded:      {CONFIG_PATH}")
print(f"data_path:          {config['data_path']}")
print(f"results_out_path:   {config.get('results_out_path', '(not set — results will not be saved to disk)')}")
print()
print(f"reg_type:           {reg_type}")
print(f"epochs:             {tcfg.get('epochs', 100)}")
print(f"batch_size:         {tcfg.get('batch_size', 32)}")
print(f"lr:                 {tcfg.get('lr', 1e-4)}")
print(f"mask_ratio:         {tcfg.get('mask_ratio', 0.5)}")
print(f"ema_momentum:       {tcfg.get('ema_momentum', 0.996)}")
if reg_type == "vicreg":
    print(f"vic_sim:            {tcfg.get('vic_sim', 25.0)}")
    print(f"vic_std:            {tcfg.get('vic_std', 25.0)}")
    print(f"vic_cov:            {tcfg.get('vic_cov', 1.0)}")
else:
    print(f"lambd (SIGReg):     {tcfg.get('lambd', 0.05)}")
    print(f"num_slices:         {tcfg.get('num_slices', 256)}")
print(f"unit_dropout:       {tcfg.get('unit_dropout', 0.0)}")
print(f"unit_id_steps:      {tcfg.get('unit_id_steps', 200)}")
print(f"unit_id_lr:         {tcfg.get('unit_id_lr', 1e-3)}")
print()
print(f"d_model:            {mcfg.get('d_model', 256)}")
print(f"n_latents:          {mcfg.get('n_latents', 64)}")
print(f"window_size_s:      {mcfg.get('window_size_s', 0.4)}")
print(f"rope_t_min:         {mcfg.get('rope_t_min', 1e-3)}")
print(f"rope_t_max:         {mcfg.get('rope_t_max', 4.0)}")
print(f"use_delimiter_tokens: {mcfg.get('use_delimiter_tokens', True)}")

In [ ]:
# Path to your checkpoint (Update this if your zip structure is different)
# If you unzipped LeJEPA_export.zip, the file is usually lejepa_checkpoint.pt
CKPT_PATH = Path("/content/lejepa_model/results/lejepa_checkpoint.pt") 
LOAD_CKPT = True

if LOAD_CKPT and CKPT_PATH.exists():
    # Reconstruct the model bundle, config, and unit_maps from the file
    # This overwrites the 'jepa_model' and 'config' variables with the saved versions
    jepa_model, config, unit_maps = load_checkpoint(CKPT_PATH)
    print(f"Successfully loaded March 11 weights from {CKPT_PATH}")
    
    # Set to eval mode to ensure stable latents for distillation
    jepa_model['context_encoder'].eval()
    jepa_model['predictor'].eval()
else:
    print(f"ERROR: Checkpoint not found at {CKPT_PATH}. Check your unzip path.") 

## Data Loading

Loads the Parquet from `data_path`, validates schema and integrity, then splits windows by `session_id` into train / val / test sets.

In [ ]:
train_loader, val_loader, test_loader, unit_maps, test_session_ids = load_and_prepare_data(config)

print(f"\nSessions in unit_maps: {len(unit_maps)}")
print(f"Units per session:     min={min(len(m) for m in unit_maps.values())}, "
      f"max={max(len(m) for m in unit_maps.values())}")
print(f"Train batches:         {len(train_loader)}")
print(f"Val batches:           {len(val_loader)}")
print(f"Test batches:          {len(test_loader)}")
print(f"Test session IDs:      {test_session_ids}")

## Training

Trains the **context encoder** (online, gradients flow), **target encoder** (EMA copy, no gradients), and **predictor** (narrow Transformer).

Regularization is selected by `reg_type` in your config:
- **SIGReg** (default): `total = (1 - λ) * MSE(h_pred, h_tgt) + λ * SIGReg(h_ctx, h_tgt)`
- **VICReg**: `total = vic_sim * MSE(h_pred, h_tgt) + vic_std * var_loss + vic_cov * cov_loss`

After training, the checkpoint is saved to `<results_out_path>/lejepa_checkpoint.pt`.

In [ ]:
if not LOAD_CKPT:
    jepa_model, jepa_train_metrics = train_lejepa(config, train_loader, val_loader, unit_maps)
    print("\nTraining complete.")
    jepa_train_metrics.tail()
else:
    print("Skipping training, checkpoint was loaded.")

## Training Results

Saves `metrics.csv` and `training_curves.png` to `<results_out_path>/LeJEPA/training/`, then renders the curves inline.

In [ ]:
if not LOAD_CKPT:
    save_results(stage="LeJEPA", phase="training", metrics=jepa_train_metrics, config=config)

    !zip -r LeJEPA_export.zip results
    files.download('LeJEPA_export.zip')
else:
    print("Skipping results saving and export, checkpoint was loaded.")

In [ ]:
if not LOAD_CKPT:
    # Inline training curves
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle("LeJEPA - Training Curves")

    axes[0].plot(jepa_train_metrics["epoch"], jepa_train_metrics["train_loss"], label="train")
    axes[0].plot(jepa_train_metrics["epoch"], jepa_train_metrics["val_loss"], label="val")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].set_title("Total Loss")
    axes[0].legend()

    axes[1].plot(jepa_train_metrics["epoch"], jepa_train_metrics["train_pred_loss"], label="pred loss")
    axes[1].plot(jepa_train_metrics["epoch"], jepa_train_metrics["train_reg_loss"], label="reg loss")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[1].set_title("Pred vs Reg Loss")
    axes[1].legend()

    plt.tight_layout()
    plt.show()
else:
    print("Skipping training curves, checkpoint was loaded.")

In [ ]:
if not LOAD_CKPT:
    # Full per-epoch metrics table
    jepa_train_metrics
else:
    print("Skipping training metrics display, checkpoint was loaded.")

## LeJEPA Test Evaluation

Runs the trained model on the held-out test set (no masking — context encoder sees all events).

Metrics per batch:
- **`pred_loss`**: MSE between predicted and target mean-pooled representations `[B, D]`
- **`cos_similarity`**: cosine similarity between context and target representations

## Unit Identification

Adapts the test-session unit embedding tables using the self-supervised LeJEPA objective, with all shared weights frozen.

The test sessions' unit embeddings were randomly initialized during training and never updated (no test data was seen during training). Unit identification fixes this by running the JEPA prediction loss on test windows for a small number of steps — only the unit embedding parameters for test sessions are trainable. This takes ~1 minute on GPU.

After this step the test representations are meaningful rather than driven by random unit-ID fingerprints.

In [ ]:
identify_units(jepa_model, test_loader, test_session_ids, config)

In [ ]:
jepa_test_metrics = evaluate_model(jepa_model, test_loader, stage="LeJEPA")

In [ ]:
save_results(stage="LeJEPA", phase="test", metrics=jepa_test_metrics, config=config)

# Display saved plots if results_out_path is set
results_path = config.get("results_out_path")
if results_path:
    test_bar_path = Path(results_path) / "LeJEPA" / "test" / "test_metrics.png"
    latent_path   = Path(results_path) / "LeJEPA" / "test" / "latent_space.png"
    if test_bar_path.exists():
        print("Test metrics bar chart:")
        display(Image(filename=str(test_bar_path)))
    if latent_path.exists():
        print("Latent space (UMAP):")
        display(Image(filename=str(latent_path)))

## PCA baseline

In [ ]:
all_spikes = []
fixed_width = 1500 # Adjust if your sessions are smaller, but 1500 is a safe middle ground

for i, batch in enumerate(test_loader):
    # Get raw data and ensure it's 2D [Batch, Features]
    s = batch["unit_ids"].float().cpu().numpy()
    if s.ndim == 3:
        s = s.reshape(s.shape[0], -1)
    
    # Standardize width to fixed_width
    if s.shape[1] >= fixed_width:
        s = s[:, :fixed_width]
    else:
        pad = np.zeros((s.shape[0], fixed_width - s.shape[1]))
        s = np.concatenate([s, pad], axis=1)
        
    all_spikes.append(s)
    if len(all_spikes) > 15: break 

raw_spikes = np.concatenate(all_spikes, axis=0)

# PCA Benchmark
n_comps = min(raw_spikes.shape[0] - 1, 256)
pca = PCA(n_components=n_comps)
z_pca = pca.fit_transform(raw_spikes)

print(f"PCA Baseline Complete.")
print(f"Total Variance Explained: {pca.explained_variance_ratio_.sum():.4f}")

In [ ]:
snn_model, snn_metrics = distill_snn(config, jepa_model, train_loader, val_loader)
print("SNN Distillation Complete.")

### Umap probe 2

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

jepa_model['context_encoder'].eval()
all_z_teacher = []
all_sessions = []

print("Extracting Teacher Latents...")
with torch.no_grad():
    for i, batch in enumerate(test_loader):
        z, _ = jepa_model['context_encoder'](
            batch["session_ids"].to(device), 
            batch["unit_ids"].to(device), 
            batch["time_ids"].to(device), 
            batch["attention_mask"].to(device)
        )
        all_z_teacher.append(z.mean(dim=1).cpu().numpy())
        all_sessions.append(batch["session_ids"].cpu().numpy())
        if i >= 10: break 

z_teacher_np = np.concatenate(all_z_teacher, axis=0)
sessions_np = np.concatenate(all_sessions, axis=0)

print("Extracting SNN Latents...")
all_z_snn = []
with torch.no_grad():
    for i, batch in enumerate(test_loader):
        z_t, _ = jepa_model['context_encoder'](
            batch["session_ids"].to(device), 
            batch["unit_ids"].to(device), 
            batch["time_ids"].to(device), 
            batch["attention_mask"].to(device)
        )
        # SNN Forward Pass
        outputs = snn_model(z_t)
        spk, mem = outputs[0], outputs[1]
        all_z_snn.append(mem.mean(dim=1).cpu().numpy())
        if i >= 10: break

z_snn_np = np.concatenate(all_z_snn, axis=0)
print("Extraction Complete. Ready for UMAP.")

# 3. Generate Plots using your latent_space util
print("Generating Teacher UMAP...")
fig_teacher, _ = latent_space.plot_umap_by_session(z_teacher_np, sessions_np, stage="LeJEPA-Teacher")
fig_teacher.savefig("teacher_umap.png")

print("Generating SNN UMAP...")
fig_snn, _ = latent_space.plot_umap_by_session(z_snn_np, sessions_np, stage="SNN-Student")
fig_snn.savefig("snn_umap.png")

plt.show()

## Download Results

In [ ]:
!zip -r LeJEPA_export.zip results
files.download('LeJEPA_export.zip')